In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

# A URL da edição do KDMiLe
url = 'https://sol.sbc.org.br/index.php/kdmile/issue/view/1576'

# Fazemos uma requisição GET para baixar o conteúdo da página
resposta = requests.get(url)

# Verificamos se a requisição foi bem-sucedida (código 200 significa sucesso)
if resposta.status_code == 200:
    # Passamos o texto do HTML para o BeautifulSoup
    soup = BeautifulSoup(resposta.text, 'html.parser')

    dados_artigos = []

    # Encontra todos os blocos dos artigos
    blocos_artigos = soup.find_all('div', class_='obj_article_summary')

    for bloco in blocos_artigos:
        # Busca a tag <a> dentro da div 'title'
        tag_a = bloco.find('div', class_='title').find('a')

        if tag_a:
            # Extrai o título limpo e o link
            titulo = tag_a.get_text(strip=True)
            link = tag_a.get('href')

            # Adiciona os dados na lista
            dados_artigos.append({
                'Título': titulo,
                'Link': link
            })

    # Cria o DataFrame
    df_artigos = pd.DataFrame(dados_artigos)

    # Exibe as primeiras linhas do resultado
    print(df_artigos.head())

else:
    print(f"Erro ao acessar a página. Status code: {resposta.status_code}")

ConnectionError: HTTPSConnectionPool(host='sol.sbc.org.br', port=443): Max retries exceeded with url: /index.php/kdmile/issue/view/1576 (Caused by NameResolutionError("HTTPSConnection(host='sol.sbc.org.br', port=443): Failed to resolve 'sol.sbc.org.br' ([Errno -3] Temporary failure in name resolution)"))

In [ ]:
df_artigos

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

# Assumindo que você já rodou o código anterior e tem o df_artigos criado
# df_artigos = pd.DataFrame(dados_artigos)

dados_completos = []

print("Iniciando a extração dos artigos...")

for index, row in df_artigos.iterrows():
    link = row['Link']
    print(f"Extraindo: {link}")

    try:
        # Pausa de 1 segundo entre as requisições para evitar sobrecarga no servidor
        time.sleep(1)

        resposta = requests.get(link)

        if resposta.status_code == 200:
            soup = BeautifulSoup(resposta.text, 'html.parser')

            # 1. Extração do Título
            titulo_tag = soup.find('h1', class_='page_title')
            titulo = titulo_tag.get_text(strip=True) if titulo_tag else row['Título']

            # 2. Extração dos Autores
            autores_tags = soup.find_all('span', class_='name')
            # Pega o texto de cada tag de autor e junta tudo em uma string separada por vírgula
            autores = ", ".join([autor.get_text(strip=True) for autor in autores_tags])

            # 3. Extração da Data de Publicação
            data_tag = soup.find('div', class_='item published')
            data_pub = ""
            if data_tag:
                data_val = data_tag.find('div', class_='value')
                if data_val:
                    data_pub = data_val.get_text(strip=True)

            # 4. Extração do Resumo
            resumo_tag = soup.find('div', class_='item abstract')
            resumo = ""
            if resumo_tag:
                # Remove a tag <h3> que contém a palavra "Resumo" para não poluir o texto
                label = resumo_tag.find('h3', class_='label')
                if label:
                    label.extract()
                resumo = resumo_tag.get_text(strip=True)

            # Adiciona os dados coletados à nova lista
            dados_completos.append({
                'Título': titulo,
                'Autores': autores,
                'Data de Publicação': data_pub,
                'Resumo': resumo,
                'Link': link
            })
        else:
            print(f"Erro {resposta.status_code} ao acessar o link: {link}")

    except Exception as e:
        print(f"Ocorreu um erro ao processar o link {link}: {e}")

# Substitui o DataFrame original pelo novo contendo todas as informações
df_artigos_completo = pd.DataFrame(dados_completos)

print("\nExtração concluída!")
print(df_artigos_completo.head())

In [ ]:
df_artigos_completo